# Install requirements and dependencies

In [ ]:
PROJECT_PATH = "/home/jupyter-gabriel/projects/early_warning_system/early-warning-system"

In [ ]:
%pip install -r "$PROJECT_PATH/requirements.txt"

## Load kedro with magic command

This will create 3 objects available in the enviroment:
1. session (a kedro.framework.session object which is capable of creating new sessions, running pipelines or nodes)
2. context (a kedro.framework.context object which contains, among other specification, the resolve catalog)
3. catalog (an object capable of loading and saving the catalog entries defined in the catalog.yml)


In [ ]:
%load_ext kedro.ipython

In [ ]:
%cd $PROJECT_PATH
%reload_kedro .

## (Optional) Load custom functions
If you want to manually test the functions you build load them

You can manually run each step and take advantage of the Kedro Catalog Utility to load and save datasets specified in the catalog

OR, you can get rid of Kedro entirely by making manual loadings and savings knowing that each function requires its own inputs and outputs

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from early_warning_system.pipelines.utils import *
from early_warning_system.pipelines.a01_aoi_period import *
from early_warning_system.pipelines.n00_create_target import *
from early_warning_system.pipelines.n01_extract_data import *
#from early_warning_system.pipelines.n02_process_data import *
#from early_warning_system.pipelines.n03_create_triggers import *
from early_warning_system.pipelines.n04_hypertuning_model import *
from early_warning_system.pipelines.n05_visualizations import *

# Coldspell pipeline

In [ ]:
%reload_kedro
runtime_params = {
    'country': 'bolivia', # Local folder name to store data 
    #'region': 'valles',
    #'lead_id': 'hevelma_seguros', # Lead name
    #'provider': 'UCSB', # Data provider, can be 'ERA5' or 'UCSB'
    'field': 'tmin', # Short variable name, can be 'swc', 'prcp', 'tmin', 'tmax'
    'peril': 'coldspell',
    #'start_year': 2006, # (optional) Use this to override the initial year of data
    #'end_year': 2025, # (optional) Use this to override the ending year of data
    'start_date': '2010-01-01',
    'end_date': '2025-12-31',
}
session_tmin = session.create(
    runtime_params = runtime_params
)
# (Optional) Load the catalog with the context provided earlier for quick loads
catalog_tmin = session_tmin.load_context().catalog 

## Step 0. Create grid

In [ ]:
gdf_aoi = catalog_tmin.load('gdf_aoi')

In [ ]:
params_s = catalog_tmin.load('params:params_s')

In [ ]:
dict_s = get_spatial_params(gdf_aoi, params_s)
gdf_grid = create_geometry(create_grid(dict_s))

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['get_spatial_params', 'create_grid', 'create_geometry']
)

## Step 1. Create target

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['get_spatial_params', 'create_target']
)

In [ ]:
df_target = catalog_tmin.load('df_target')

## Step 2. Create features

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['create_features_topo']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['create_features_fct_tmin']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['create_features_fct_10u']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['create_features_fct_10v']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['create_features_fct_msl']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['extract']
    node_names = ['create_features_fct_swc']
)

## Step 3. Create MT

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    tags=['mt']
    #node_names = ['create_features_fct_swc']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    tags=['mt']
    #node_names = ['create_features_fct_swc']
)

## Step 4. Train model

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['mt']
    node_names = ['model_hypertuning']
)

In [ ]:
dict_tuning = catalog_tmin.load('dict_tuning')

In [ ]:
dict_tuning['metrics']

In [ ]:
dict_tuning['tuning_results']['params'].iloc[0]

In [ ]:
dict_tuning_old = catalog_tmin.load('dict_tuning')
dict_tuning_old['metrics']

## Step 5. Final model

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['mt']
    node_names = ['final_model']
)

In [ ]:
dict_final = catalog_tmin.load('dict_final')

In [ ]:
dict_final['metrics']

In [ ]:
dict_final['specifications']['params_algorithm'].iloc[0]

In [ ]:
with open('/home/jupyter-gabriel/suyana/ews_downscaling/reports/ews_downscale_temperature_tuning_summary_xgbc_target_1sd_3d_final_v5_run01_old.pkl', 'rb') as pkl:
    dict_old = pickle.load(pkl)

In [ ]:
dict_old['tuning_results'].iloc[0,:]['params']

## Step 6. Visualization

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['mt']
    node_names = ['plot_metrics']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params
).run(
    #tags=['mt']
    node_names = ['plot_metrics', 'plot_scatter', 'plot_curves']
)

# Precipitation pipeline

In [ ]:
%reload_kedro
runtime_params_prcp = {
    'country': 'bolivia', # Local folder name to store data 
    #'region': 'valles',
    #'lead_id': 'hevelma_seguros', # Lead name
    #'provider': 'UCSB', # Data provider, can be 'ERA5' or 'UCSB'
    'field': 'prcp', # Short variable name, can be 'swc', 'prcp', 'tmin', 'tmax'
    'peril': 'rainfall',
    #'start_year': 2006, # (optional) Use this to override the initial year of data
    #'end_year': 2025, # (optional) Use this to override the ending year of data
    'start_date': '2010-01-01',
    'end_date': '2025-12-31',
}
session_prcp = session.create(
    runtime_params = runtime_params_prcp
)
# (Optional) Load the catalog with the context provided earlier for quick loads
catalog_prcp = session_prcp.load_context().catalog 

## Step 1. Create target

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    #tags=['extract']
    node_names = ['get_spatial_params', 'create_target']
)

In [ ]:
df_target = catalog_prcp.load('df_target')

In [ ]:
import matplotlib.dates as mdates
def plot_curation(df, station_id, time_start='2024-01-01', time_end=None,
                  col_original='tmin_lfa', col_auxiliary='tmin_chirts',
                  col_final='tmin', var_label='TMIN (°C)',
                  label_original='Original (LFA)',
                  label_auxiliary='Auxiliary (CHIRTS)',
                  label_final='Curated (final)',
                  figsize=(11, 4.5), title=None):
    """
    Visualize how the target series was curated using an auxiliary dataset.

    Parameters
    ----------
    df : DataFrame
        Must contain 'idStation', 'time', and the three value columns below.
    station_id : int or str
        Station to plot.
    time_start, time_end : str or Timestamp, optional
        Time window. time_end is inclusive.
    col_original, col_auxiliary, col_final : str
        Column names for the original, auxiliary and curated series.
    var_label : str
        Y axis label (units).
    label_original, label_auxiliary, label_final : str
        Legend labels for each line.

    Layers
    ------
    1. Auxiliary: reference background.
    2. Original: raw station record, gaps and outliers visible.
    3. Final: curated series used downstream.
    4. Shaded spans where the final value came from the auxiliary source
       (original missing but final present).
    """
    mask = (df['idStation'] == station_id) & (df['time'] >= time_start)
    if time_end is not None:
        mask &= df['time'] <= time_end
    d = df.loc[mask].sort_values('time').copy()

    filled = d[col_original].isna() & d[col_final].notna()

    fig, ax = plt.subplots(1, 1, figsize=figsize)

    ax.plot(d['time'], d[col_auxiliary],
            color='#7f7f7f', linestyle=':', linewidth=1.0, alpha=0.7,
            label=label_auxiliary, zorder=1)

    ax.plot(d['time'], d[col_original],
            color='#1f77b4', linestyle='-', linewidth=1.0, alpha=0.55,
            label=label_original, zorder=2)

    ax.plot(d['time'], d[col_final],
            color='#d62728', linestyle='-', linewidth=1.6, alpha=0.95,
            label=label_final, zorder=3)

    if filled.any():
        t = d['time'].values
        in_gap = False
        gap_start = None
        first_span = True
        for i, flag in enumerate(filled.values):
            if flag and not in_gap:
                gap_start = t[i]
                in_gap = True
            elif not flag and in_gap:
                ax.axvspan(gap_start, t[i], color='#ffcf6b', alpha=0.25,
                           zorder=0,
                           label='Filled from auxiliary' if first_span else None)
                first_span = False
                in_gap = False
        if in_gap:
            ax.axvspan(gap_start, t[-1], color='#ffcf6b', alpha=0.25,
                       zorder=0,
                       label='Filled from auxiliary' if first_span else None)

    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y %b'))
    ax.tick_params(axis='x', labelcolor='gray', labelsize=8)
    ax.tick_params(axis='y', labelcolor='gray', labelsize=8)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)
    for spine in ['left', 'bottom']:
        ax.spines[spine].set_edgecolor('gray')
        ax.spines[spine].set_linewidth(0.5)

    ax.set_xlabel('DATE', color='gray', fontsize=9)
    ax.set_ylabel(var_label, color='gray', fontsize=9)

    n_total = len(d)
    n_filled = int(filled.sum())
    pct = (n_filled / n_total * 100) if n_total else 0
    head = title if title is not None else 'TIME SERIES CURATION'
    ax.set_title(f'{head}', color='black', fontweight='bold',
                 size=11, loc='left')

    ax.legend(loc='best', frameon=False, fontsize=9, ncol=4)
    ax.grid(True, axis='y', linestyle=':', linewidth=0.4, alpha=0.5)

    plt.tight_layout()
    plt.close()
    return fig


In [ ]:
df_target['idStation'].unique()

In [ ]:
plot_curation(
    df_target, station_id=751,
    time_start='2024-01-01',
    col_original='prcp_lfa', 
    col_auxiliary='prcp_chirps',
    col_final='prcp', var_label='TMIN (°C)',
    label_original='Original (LFA)',
    label_auxiliary='Auxiliary (CHIRTS)',
    label_final='Curated (final)',
    title='TIME SERIES CURATION FOR COLDSPELLS'
)

In [ ]:
df_target['idStation'].unique()

In [ ]:
df_target.columns

In [ ]:
df_plot = df_target[df_target['idStation']==937].copy()

In [ ]:
df_plot['yearmon'] = df_plot['time'].dt.strftime('%Y%m')
df_plot = df_plot.groupby(['idStation', 'yearmon'], as_index=False).agg(
    prcp_chirps = ('prcp_chirps', 'sum'),
    prcp_lfa = ('prcp_lfa', 'sum')
)
df_plot['time'] = pd.to_datetime(df_plot['yearmon'], format='%Y%m')

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 7))

ax.scatter(df_plot['prcp_chirps'], df_plot['prcp_lfa'], alpha=0.4, s=10)

lims = [
    min(ax.get_xlim()[0], ax.get_ylim()[0]),
    max(ax.get_xlim()[1], ax.get_ylim()[1]),
]
ax.plot(lims, lims, color='red', linewidth=1, linestyle='--', label='1:1')
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel('CHIRPS')
ax.set_ylabel('Station (LFA)')
ax.legend()

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14,4))

#ax.plot(df_plot['time'], df_plot['prcp_chirps'], linewidth=1, color='red', alpha=0.5)
ax.plot(df_plot['time'], df_plot['prcp'], linewidth=1, color='darkorange', alpha=0.6)
ax.plot(df_plot['time'], df_plot['prcp_lfa'], linewidth=1, color='darkgreen', alpha=0.4)
#ax.scatter(df_plot['prcp_chirps'], df_plot['prcp_lfa'])

## Step 2. Create additional features

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    #tags=['extract']
    node_names = ['create_features_fct_tp']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    #tags=['extract']
    node_names = ['create_features_fct_tp_chirps']
)

## Step 3. Create MT

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    tags=['mt']
    #node_names = ['create_features_fct_swc']
)

## Step 4. Train model

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    #tags=['mt']
    node_names = ['model_hypertuning']
)

In [ ]:
dict_tuning = catalog_prcp.load('dict_tuning')

In [ ]:
dict_tuning['tuning_results']['params'].iloc[0]

In [ ]:
dict_tuning['metrics']

## Step 5. Final model

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    #tags=['mt']
    node_names = ['final_model']
)

In [ ]:
dict_final = catalog_prcp.load('dict_final')

In [ ]:
dict_final['metrics']

In [ ]:
dict_final_old = catalog_prcp.load('dict_final')

In [ ]:
dict_final_old['metrics']

## Step 6. Visualization

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    #tags=['mt']
    node_names = ['plot_metrics', 'plot_scatter', 'plot_curves']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    #tags=['mt']
    node_names = ['plot_scatter']
)

In [ ]:
%reload_kedro 
session.create(
    runtime_params=runtime_params_prcp
).run(
    pipeline_names = ['rainfall'],
    #tags=['mt']
    node_names = ['plot_curves']
)

In [ ]:
#TODO: Update hypertuning to maximize the gini in both backtest and out-of-sample

# Test datasets

In [ ]:
catalog_tmin.load('ds_f_dem')

In [ ]:
df_target['target_1sd_3d_final'].value_counts(normalize=True)

In [ ]:
catalog_tmin.load('ds_aux_tmin')

In [ ]:
catalog_tmin.load('ds_aux_tmin')

In [ ]:
ds_dem = catalog_tmin.load('ds_dem')

In [ ]:
import matplotlib.pyplot as plt
plt.pcolormesh(ds_dem['lon'], ds_dem['lat'], ds_dem['altitude'])

In [ ]:
ds_fct

In [ ]:
ds_fct = catalog_tmin.load('ds_aux_tmin')

In [ ]:
ds_10u_plot = ds_fct.isel(time=1)#, step=1)

plt.pcolormesh(ds_10u_plot['lon'], ds_10u_plot['lat'], ds_10u_plot['tmin'])